# 📊 Machine Learning Interactive Visualizer
## Linear Regression · Sales Forecasting for **ABC Electronics**

> **A teaching notebook, not a code dump.** Each lesson lives in its **own cell**.
> Run a cell → an animated, narrated scene appears → discuss it → run the next cell.

---

### ▶ How to teach with this notebook
1. Run the **4 setup cells** at the top once (`Runtime → Run all` is easiest, or run them one by one).
2. Then run the scene cells **one at a time**, top to bottom. Each cell = **one concept, one animation, one lesson**.
3. Inside every scene you get:
   - a **↻ Replay** button (re-plays the animation) and a **🔊 Narration** button (spoken explanation),
   - an interactive visualization,
   - a **Key Takeaway**.

### 🧭 Lesson map (17 scenes)
| # | Lesson | # | Lesson |
|---|--------|---|--------|
| 1 | Welcome | 10 | **🔬 Inside the model — one data point's journey** |
| 2 | The business story | 11 | **🎯 Finding the best-fit line (you try it!)** |
| 3 | Meet the dataset | 12 | Watch the model learn |
| 4 | Understand the numbers | 13 | Testing on unseen data |
| 5 | Table → picture | 14 | Forecast the future (slider) |
| 6 | Finding the relationship | 15 | Business recommendations |
| 7 | Choosing the feature | 16 | Explain why |
| 8 | Choosing the target | 17 | Summary |
| 9 | Train / test split | | |

*Built with Python · NumPy · Pandas · Plotly · ipywidgets · scikit-learn · gTTS — 100% inside Google Colab.*


In [ ]:
# @title 🔧 Step 1 — Install & enable the interactive engine  { display-mode: "form" }
# ------------------------------------------------------------------
#  Installs the (few) libraries Colab does not ship by default and
#  switches on the widget manager so buttons / sliders render live.
# ------------------------------------------------------------------
import sys, subprocess

def _pip(*pkgs):
    """Quietly install packages, ignoring ones already present."""
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", *pkgs],
        check=False,
    )

# gTTS = Google Text-to-Speech for narration. plotly/ipywidgets/sklearn
# are usually pre-installed on Colab but we pin them just in case.
_pip("gTTS", "plotly>=5.18", "ipywidgets>=7.7", "scikit-learn")

# Enable rich widgets inside Colab output cells.
try:
    from google.colab import output as _colab_output
    _colab_output.enable_custom_widget_manager()
    print("✅ Colab custom widget manager enabled.")
except Exception:
    print("ℹ️ Not running inside Colab — widgets will still work in classic Jupyter.")

print("✅ Environment ready. Run the remaining cells top-to-bottom.")


In [ ]:
# @title 🎨 Step 2 — Imports, design system & animation stylesheet
# ------------------------------------------------------------------
#  A single place that defines the visual language of the whole app:
#  colours, fonts, CSS keyframe animations and a Plotly theme helper.
# ------------------------------------------------------------------
from __future__ import annotations
from typing import Callable

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
import ipywidgets as widgets
from IPython.display import display, HTML, Audio

# Plotly needs the dedicated Colab renderer to draw inside widget outputs.
pio.renderers.default = "colab"

# ---- Colour palette (one source of truth) ------------------------
THEME = {
    "bg":      "#0F1226",
    "panel":   "#1B1F3B",
    "panel2":  "#232850",
    "text":    "#EAECFF",
    "muted":   "#A6ADD8",
    "accent":  "#6C5CE7",   # violet
    "accent2": "#00D1B2",   # teal
    "gold":    "#FFD166",
    "good":    "#2ECC71",
    "bad":     "#FF6B6B",
    "grid":    "#2C3160",
}

# ---- Global stylesheet: injected once, reused by every scene -----
GLOBAL_CSS = """
<style>
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;600;800&display=swap');
.mlv-wrap, .mlv-wrap * { box-sizing: border-box; font-family:'Inter','Segoe UI',sans-serif; }
.mlv-wrap { color:#EAECFF; }

.mlv-kicker { color:#00D1B2; font-weight:800; letter-spacing:3px; text-transform:uppercase; font-size:12px; margin-top:4px; }
.mlv-title  { font-size:30px; font-weight:800; margin:2px 0 10px;
              background:linear-gradient(90deg,#ffffff,#B9B4FF); -webkit-background-clip:text; background-clip:text; color:transparent;
              animation:fadeUp .6s ease both; }
.mlv-question { background:linear-gradient(90deg,rgba(108,92,231,.28),rgba(0,209,178,.15));
                border:1px solid #6C5CE7; border-radius:14px; padding:12px 16px; font-size:16px; font-weight:600;
                animation:fadeUp .6s .1s ease both; }

.mlv-prog { margin:2px 0 14px; }
.mlv-prog-track { height:8px; background:#232850; border-radius:99px; overflow:hidden; }
.mlv-prog-fill  { height:100%; background:linear-gradient(90deg,#6C5CE7,#00D1B2); border-radius:99px;
                  transition:width .6s ease; box-shadow:0 0 12px #6C5CE7; }
.mlv-dots { display:flex; gap:6px; flex-wrap:wrap; margin-top:8px; }
.mlv-dot  { width:9px; height:9px; border-radius:50%; background:#232850; transition:.3s; }
.mlv-dot.done { background:#00D1B2; }
.mlv-dot.cur  { background:#FFD166; transform:scale(1.5); box-shadow:0 0 10px #FFD166; }
.mlv-prog-label { color:#A6ADD8; font-size:12px; margin-top:6px; }

.mlv-takeaway { background:linear-gradient(90deg,rgba(255,209,102,.16),rgba(255,209,102,.04));
                border-left:5px solid #FFD166; border-radius:12px; padding:14px 18px; font-size:15px; margin-top:16px;
                animation:fadeUp .6s ease both; }

.mlv-grid { display:flex; flex-wrap:wrap; gap:14px; margin:10px 0; }
.mlv-card { background:#232850; border:1px solid #2C3160; border-radius:16px; padding:16px 18px; flex:1; min-width:170px;
            animation:fadeUp .6s ease both; box-shadow:0 8px 24px rgba(0,0,0,.28); }
.mlv-card h4 { margin:0 0 6px; color:#00D1B2; font-size:12px; text-transform:uppercase; letter-spacing:1px; }
.mlv-stat { font-size:28px; font-weight:800; }
.mlv-sub  { color:#A6ADD8; font-size:13.5px; line-height:1.5; }
.mlv-badge { display:inline-block; padding:3px 10px; border-radius:99px; background:#6C5CE7; color:#fff; font-size:12px; font-weight:700; }

.mlv-flow { display:flex; align-items:center; gap:10px; flex-wrap:wrap; justify-content:center; margin:18px 0; }
.mlv-box  { background:#232850; border:2px solid #6C5CE7; border-radius:14px; padding:14px 18px; font-weight:700;
            text-align:center; min-width:110px; animation:pop .5s ease both; }
.mlv-box small { display:block; color:#A6ADD8; font-weight:400; font-size:11px; margin-top:4px; }
.mlv-box.model { border-color:#FFD166; background:linear-gradient(180deg,#232850,#2a2f5e); }
.mlv-box.err   { border-color:#FF6B6B; }
.mlv-arrow { font-size:26px; color:#00D1B2; animation:dash 1.2s ease-in-out infinite; }
.mlv-gear  { display:inline-block; font-size:30px; animation:spin 3s linear infinite; }
.mlv-gear.r{ animation:spinr 2.2s linear infinite; }

.mlv-table { width:100%; border-collapse:collapse; margin-top:10px; font-size:14px; }
.mlv-table th { background:#6C5CE7; color:#fff; padding:8px 12px; text-align:left; position:sticky; top:0; }
.mlv-table td { padding:7px 12px; border-bottom:1px solid #2C3160; }
.mlv-table tr { animation:rowIn .45s ease both; }
.mlv-hl   { background:rgba(255,209,102,.18) !important; }
.mlv-fade { opacity:.25; }

.mlv-meter { height:16px; border-radius:99px; background:#232850; overflow:hidden; margin-top:8px; }
.mlv-meter > div { height:100%; background:linear-gradient(90deg,#6C5CE7,#2ECC71); border-radius:99px; animation:grow 1.3s ease both; }

.mlv-float { animation:floaty 3s ease-in-out infinite; }

/* --- Data-flow pipeline (Scene 10: one point's journey) --- */
.mlv-pipe  { display:flex; align-items:center; gap:8px; flex-wrap:wrap; justify-content:center; margin:14px 0; }
.mlv-stage { background:#232850; border:2px solid #6C5CE7; border-radius:14px; padding:10px 16px; text-align:center;
             min-width:86px; animation:pop .5s ease both; }
.mlv-stage .op  { font-size:11px; color:#A6ADD8; text-transform:uppercase; letter-spacing:1px; }
.mlv-stage .val { font-size:24px; font-weight:800; color:#FFD166; }
.mlv-stage.result { border-color:#FFD166; box-shadow:0 0 16px rgba(255,209,102,.35); animation:pop .5s ease both, glow 1.8s ease-in-out infinite; }
.mlv-stage.err    { border-color:#FF6B6B; }
.mlv-op { font-size:22px; font-weight:800; color:#00D1B2; animation:pop .5s ease both; }

@keyframes glow   { 0%,100%{box-shadow:0 0 12px rgba(255,209,102,.3)} 50%{box-shadow:0 0 24px rgba(255,209,102,.6)} }
@keyframes fadeUp { from{opacity:0; transform:translateY(16px)} to{opacity:1; transform:none} }
@keyframes pop    { 0%{opacity:0; transform:scale(.7)} 70%{transform:scale(1.06)} 100%{opacity:1; transform:scale(1)} }
@keyframes grow   { from{width:0} to{width:var(--w,100%)} }
@keyframes spin   { to{transform:rotate(360deg)} }
@keyframes spinr  { to{transform:rotate(-360deg)} }
@keyframes dash   { 0%,100%{transform:translateX(0); opacity:.5} 50%{transform:translateX(6px); opacity:1} }
@keyframes rowIn  { from{opacity:0; transform:translateX(-14px)} to{opacity:1; transform:none} }
@keyframes floaty { 0%,100%{transform:translateY(0)} 50%{transform:translateY(-8px)} }
</style>
"""
display(HTML(GLOBAL_CSS))


def plotly_theme(fig: go.Figure, height: int = 460, title: str | None = None) -> go.Figure:
    """Apply the app's dark visual language to any Plotly figure."""
    fig.update_layout(
        template="plotly_dark",
        paper_bgcolor=THEME["panel"],
        plot_bgcolor=THEME["panel"],
        font=dict(color=THEME["text"], family="Inter, Segoe UI, sans-serif", size=14),
        margin=dict(l=60, r=30, t=64 if title else 30, b=52),
        height=height,
        title=dict(text=title or "", font=dict(size=19, color=THEME["text"]), x=0.02),
        legend=dict(bgcolor="rgba(0,0,0,0)", orientation="h", y=1.08, x=0.0),
    )
    fig.update_xaxes(gridcolor=THEME["grid"], zerolinecolor=THEME["grid"])
    fig.update_yaxes(gridcolor=THEME["grid"], zerolinecolor=THEME["grid"])
    return fig


def play_menu(duration: int = 140) -> dict:
    """Reusable Plotly ▶ Play / ⏮ Reset control for frame animations."""
    return dict(
        type="buttons", showactive=False, x=0.02, y=1.16, xanchor="left",
        bgcolor=THEME["panel2"], font=dict(color=THEME["text"]),
        buttons=[
            dict(label="▶ Play", method="animate",
                 args=[None, dict(frame=dict(duration=duration, redraw=True),
                                  fromcurrent=True, transition=dict(duration=0))]),
            dict(label="⏮ Reset", method="animate",
                 args=[["0"], dict(frame=dict(duration=0, redraw=True), mode="immediate")]),
        ],
    )

print("✅ Design system loaded.")


In [ ]:
# @title 📦 Step 3 — Business data & the trained model
# ------------------------------------------------------------------
#  ABC Electronics' historical monthly sales, plus a Linear Regression
#  model trained on it. Everything downstream reads from here.
# ------------------------------------------------------------------
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error


def generate_dataset(seed: int = 42) -> pd.DataFrame:
    """Create 24 months of realistic, upward-trending sales (in thousands of units)."""
    rng = np.random.default_rng(seed)
    months = np.arange(1, 25)
    true_intercept, true_slope = 52.0, 4.3          # the "hidden truth" we hope to recover
    noise = rng.normal(0, 6.5, size=months.size)    # real-world randomness
    sales = true_intercept + true_slope * months + noise
    return pd.DataFrame({"Month": months, "Sales": np.round(sales, 1)})


# ---- Data -------------------------------------------------------
df: pd.DataFrame = generate_dataset()
X: np.ndarray = df[["Month"]].values            # feature matrix (2-D)
y: np.ndarray = df["Sales"].values              # target vector

# ---- Correlation (how strongly Month & Sales move together) -----
CORR: float = float(np.corrcoef(df["Month"], df["Sales"])[0, 1])

# ---- Train / test split -----------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=7, shuffle=True
)

# ---- Model ------------------------------------------------------
model: LinearRegression = LinearRegression().fit(X_train, y_train)
SLOPE: float = float(model.coef_[0])
INTERCEPT: float = float(model.intercept_)

# ---- Metrics on the held-out test set ---------------------------
_y_pred_test = model.predict(X_test)
R2:   float = float(r2_score(y_test, _y_pred_test))
MAE:  float = float(mean_absolute_error(y_test, _y_pred_test))
RMSE: float = float(np.sqrt(mean_squared_error(y_test, _y_pred_test)))

# ---- Handy summary stats ----------------------------------------
STATS = df["Sales"].describe()


def predict_sales(month: int) -> float:
    """Predict sales (k units) for a given month number using the trained model."""
    return float(model.predict(np.array([[month]]))[0])


print(f"✅ Data & model ready — Sales ≈ {INTERCEPT:.1f} + {SLOPE:.2f} × Month "
      f"| corr={CORR:.3f} | test R²={R2:.3f}")


In [ ]:
# @title 🧩 Step 4 — Narration, UI components & the per-scene `teach()` renderer
# ==================================================================
#  This is the last setup cell. It provides:
#    • voice narration (gTTS, cached, offline-safe)
#    • small HTML building blocks (cards, tables, progress bar)
#    • teach(...) — the function every scene cell calls to render a
#      standalone lesson: header + animation + takeaway + ↻Replay + 🔊Narrate
# ==================================================================
import io

# ---------------- Narration (gTTS, cached) ------------------------
_AUDIO_CACHE: dict[str, bytes | None] = {}


def synth_narration(key: str, text: str) -> bytes | None:
    """Return MP3 bytes for `text`, generating & caching on first use."""
    if key in _AUDIO_CACHE:
        return _AUDIO_CACHE[key]
    try:
        from gtts import gTTS
        buf = io.BytesIO()
        gTTS(text=text, lang="en", slow=False).write_to_fp(buf)
        data = buf.getvalue()
    except Exception as exc:
        print("⚠️ Narration unavailable:", exc)
        data = None
    _AUDIO_CACHE[key] = data
    return data


def play_narration_into(out: widgets.Output, key: str, text: str) -> None:
    out.clear_output(wait=True)
    with out:
        display(HTML('<div class="mlv-wrap"><span class="mlv-sub">🔊 Generating narration…</span></div>'))
    data = synth_narration(key, text)
    out.clear_output(wait=True)
    with out:
        if data:
            display(Audio(data=data, autoplay=True))
        else:
            display(HTML('<div class="mlv-wrap"><span class="mlv-sub">🔇 Narration needs internet '
                         '(gTTS). Read the on-screen text instead.</span></div>'))


# ---------------- HTML building blocks ----------------------------
def progress_html(idx: int, total: int) -> str:
    pct = (idx / (total - 1) * 100) if total > 1 else 100
    dots = "".join(
        f'<span class="mlv-dot {"done" if i < idx else ("cur" if i == idx else "todo")}"></span>'
        for i in range(total)
    )
    return (f'<div class="mlv-prog"><div class="mlv-prog-track">'
            f'<div class="mlv-prog-fill" style="width:{pct:.0f}%"></div></div>'
            f'<div class="mlv-dots">{dots}</div>'
            f'<div class="mlv-prog-label">Lesson {idx + 1} of {total}</div></div>')


def header_html(step: int, total: int, kicker: str, title: str, question: str) -> str:
    return (f'<div class="mlv-wrap">{progress_html(step - 1, total)}'
            f'<div class="mlv-kicker">{kicker}</div>'
            f'<div class="mlv-title">{title}</div>'
            f'<div class="mlv-question">🎯 &nbsp;{question}</div></div>')


def takeaway_html(text: str) -> str:
    return (f'<div class="mlv-wrap"><div class="mlv-takeaway">'
            f'<b>💡 Key Takeaway</b><br>{text}</div></div>')


def stat_card(label: str, value: str, sub: str = "", delay: float = 0.0, color: str = "#EAECFF") -> str:
    return (f'<div class="mlv-card" style="animation-delay:{delay:.2f}s"><h4>{label}</h4>'
            f'<div class="mlv-stat" style="color:{color}">{value}</div>'
            f'<div class="mlv-sub">{sub}</div></div>')


def info_card(title: str, body: str, delay: float = 0.0) -> str:
    return (f'<div class="mlv-card" style="animation-delay:{delay:.2f}s"><h4>{title}</h4>'
            f'<div class="mlv-sub">{body}</div></div>')


def grid(*cards: str) -> str:
    return f'<div class="mlv-wrap"><div class="mlv-grid">{"".join(cards)}</div></div>'


def data_table(frame: pd.DataFrame, highlight: str | None = None,
               fade_others: bool = False, max_rows: int = 24) -> str:
    cols = list(frame.columns)
    head = "".join(f'<th class="{"mlv-hl" if c == highlight else ""}">{c}</th>' for c in cols)
    rows = []
    for r, (_, row) in enumerate(frame.head(max_rows).iterrows()):
        cells = ""
        for c in cols:
            cls = "mlv-hl" if c == highlight else ("mlv-fade" if (fade_others and c != highlight) else "")
            cells += f'<td class="{cls}">{row[c]}</td>'
        rows.append(f'<tr style="animation-delay:{r * 0.05:.2f}s">{cells}</tr>')
    return (f'<div class="mlv-wrap" style="max-height:360px;overflow:auto;border-radius:14px;border:1px solid #2C3160">'
            f'<table class="mlv-table"><thead><tr>{head}</tr></thead>'
            f'<tbody>{"".join(rows)}</tbody></table></div>')


def show_html(html: str) -> None:
    display(HTML(html))


def show_fig(fig: go.Figure) -> None:
    fig.show()


# ---------------- The per-scene renderer --------------------------
def teach(step: int, total: int, kicker: str, title: str, question: str,
          narration: str, takeaway: str, body: Callable[[], None]) -> None:
    """Render ONE standalone teaching scene inside its own notebook cell.

    body() should DISPLAY the scene's animation / figure / widgets. The header,
    takeaway, replay and narration controls are added automatically.
    """
    stage = widgets.Output()
    narr = widgets.Output()

    def render_body() -> None:
        stage.clear_output(wait=True)
        with stage:
            body()
            show_html(takeaway_html(takeaway))

    b_replay = widgets.Button(description="↻ Replay", layout=widgets.Layout(width="120px"))
    b_narrate = widgets.Button(description="🔊 Narration", layout=widgets.Layout(width="130px"))
    b_replay.style.button_color = THEME["panel2"]
    b_narrate.style.button_color = THEME["accent2"]
    b_replay.on_click(lambda _: render_body())
    b_narrate.on_click(lambda _: play_narration_into(narr, f"scene-{step}", narration))

    show_html(header_html(step, total, kicker, title, question))
    display(widgets.HBox([b_replay, b_narrate],
                         layout=widgets.Layout(gap="10px", margin="4px 0 8px")))
    display(stage, narr)
    render_body()

TOTAL_SCENES = 17
print("✅ Narration, UI components and the teach() renderer are ready. "
      "Now run the scene cells below, one at a time.")


In [ ]:
# @title 🎬 Scene 1 · Welcome    ▶ run me
# ============================================================
#  ONE lesson: what this course is and why it matters.
# ============================================================
def scene_body():
    show_html('<div class="mlv-wrap"><div class="mlv-title mlv-float" style="font-size:40px">'
              '📈 Machine Learning, brought to life</div>'
              '<div class="mlv-sub" style="font-size:16px">A guided, animated journey through '
              '<b>Linear Regression</b> using a real business problem.</div></div>')
    show_html(grid(
        info_card("You will learn", "How a straight line can <b>predict the future</b> from past data.", 0.05),
        info_card("The business", "Sales forecasting for <b>ABC Electronics</b>.", 0.15),
        info_card("The method", "Discovery, not memorisation — every idea is <b>animated</b>.", 0.25),
        info_card("By the end", "You'll understand the <b>WHY, HOW & WHEN</b> of regression.", 0.35),
    ))
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=df["Month"], y=df["Sales"], mode="lines+markers",
                             line=dict(color=THEME["accent2"], width=3),
                             marker=dict(size=7, color=THEME["gold"]), name="Monthly sales"))
    plotly_theme(fig, title="ABC Electronics — 24 months of sales (k units)")
    show_fig(fig)


teach(step=1, total=TOTAL_SCENES, kicker="Welcome",
      title="Machine Learning Interactive Visualizer",
      question="What are we about to learn, and why does it matter?",
      narration="Welcome to the Machine Learning Interactive Visualizer. Over the next few minutes you'll "
                "discover how a single straight line can predict the future for a real business. This isn't a "
                "coding lesson — it's a guided, animated journey into how linear regression really works.",
      takeaway="Linear Regression turns past data into future predictions — and you're about to see exactly "
               "how, step by step.",
      body=scene_body)


In [ ]:
# @title 🎬 Scene 2 · The Business Story    ▶ run me
# ============================================================
#  ONE lesson: the real business question that starts every ML project.
# ============================================================
def scene_body():
    total = df["Sales"].sum()
    avg = df["Sales"].mean()
    growth = df["Sales"].iloc[-3:].mean() - df["Sales"].iloc[:3].mean()
    show_html(grid(
        stat_card("Total sales (24 mo)", f"{total:,.0f}k", "units shipped", 0.05, THEME["accent2"]),
        stat_card("Average / month", f"{avg:,.1f}k", "units", 0.15, THEME["gold"]),
        stat_card("Growth trend", f"+{growth:,.0f}k", "recent vs. early months", 0.25, THEME["good"]),
    ))
    show_html('<div class="mlv-wrap"><div class="mlv-card" style="animation-delay:.3s">'
              '<h4>The manager\'s question</h4><div class="mlv-sub">“Our sales keep climbing. I need to know '
              '<b>how many units we\'ll sell next month</b> so I can order stock and plan marketing. '
              'Can the data tell us?”</div></div></div>')
    fig = go.Figure(go.Bar(x=df["Month"], y=df["Sales"],
                           marker=dict(color=df["Sales"], colorscale="Tealgrn", showscale=False)))
    fig.update_xaxes(title="Month"); fig.update_yaxes(title="Sales (k units)")
    plotly_theme(fig, title="Historical monthly sales")
    show_fig(fig)


teach(step=2, total=TOTAL_SCENES, kicker="Business Story", title="Meet ABC Electronics",
      question="What real problem are we trying to solve?",
      narration="ABC Electronics has sold electronics for two years, and sales keep climbing. The manager has "
                "one urgent question: how many units will we sell next month? The answer decides how much stock "
                "to buy and how much to spend on marketing.",
      takeaway="Every machine learning project starts with a business question. Ours: predict next month's "
               "sales for ABC Electronics.",
      body=scene_body)


In [ ]:
# @title 🎬 Scene 3 · Meet the Dataset    ▶ run me
# ============================================================
#  ONE lesson: the raw material — rows of Month and Sales.
# ============================================================
def scene_body():
    show_html('<div class="mlv-wrap"><div class="mlv-sub" style="margin-bottom:6px">'
              'Watch each record slide in. This is everything the model will learn from.</div></div>')
    show_html(data_table(df))
    show_html(grid(
        info_card("Month", "The input — a simple counter, 1 → 24. This is our <b>feature</b>.", 0.1),
        info_card("Sales", "The output we care about, in thousands of units. This is our <b>target</b>.", 0.2),
    ))


teach(step=3, total=TOTAL_SCENES, kicker="The Data", title="Meet the Dataset",
      question="What information do we actually have?",
      narration="Here is the raw data — twenty-four rows, one per month. Each row records the month number and "
                "the sales for that month. This is everything the model will learn from.",
      takeaway="Our data is beautifully simple: a Month column and a Sales column, 24 rows in total.",
      body=scene_body)


In [ ]:
# @title 🎬 Scene 4 · Understand the Numbers    ▶ run me
# ============================================================
#  ONE lesson: summary statistics — the shape, centre and spread.
# ============================================================
def scene_body():
    show_html(grid(
        stat_card("Rows", f"{df.shape[0]}", "monthly records", 0.05, THEME["accent2"]),
        stat_card("Columns", f"{df.shape[1]}", "Month, Sales", 0.12, THEME["gold"]),
        stat_card("Data types", "int · float", "Month, Sales", 0.19, THEME["muted"]),
    ))
    show_html(grid(
        stat_card("Mean sales", f"{STATS['mean']:.1f}k", "average month", 0.26, THEME["good"]),
        stat_card("Min → Max", f"{STATS['min']:.0f} → {STATS['max']:.0f}k", "range", 0.33, THEME["accent2"]),
        stat_card("Std dev", f"{STATS['std']:.1f}k", "spread around the mean", 0.40, THEME["gold"]),
    ))
    fig = go.Figure(go.Box(y=df["Sales"], name="Sales", boxpoints="all", jitter=0.4,
                           marker=dict(color=THEME["accent2"]), line=dict(color=THEME["gold"])))
    fig.update_yaxes(title="Sales (k units)")
    plotly_theme(fig, title="How the sales figures are spread out", height=420)
    show_fig(fig)


teach(step=4, total=TOTAL_SCENES, kicker="Statistics", title="Understanding the Numbers",
      question="What do the numbers look like as a whole?",
      narration="Before modelling, we get to know the data. Twenty-four rows, two columns, sales averaging "
                "around the middle of our range and spreading out with a clear upward drift. These summary "
                "statistics are our first health check.",
      takeaway="A quick statistical summary tells us the shape, centre and spread of the data before we "
               "model it.",
      body=scene_body)


In [ ]:
# @title 🎬 Scene 5 · Table → Picture    ▶ run me
# ============================================================
#  ONE lesson: every row becomes one point — press ▶ to reveal them.
# ============================================================
def scene_body():
    show_html('<div class="mlv-wrap"><div class="mlv-sub">Numbers in a table hide their story. Watch each row '
              'become a point: month along the bottom, sales up the side. Press ▶.</div></div>')
    x, yv = df["Month"].values, df["Sales"].values
    # One animation frame per revealed point.
    frames = [go.Frame(data=[go.Scatter(x=x[:k], y=yv[:k], mode="markers",
                                        marker=dict(color=THEME["accent2"], size=11))], name=str(k))
              for k in range(1, len(x) + 1)]
    fig = go.Figure(data=[go.Scatter(x=x[:1], y=yv[:1], mode="markers",
                                     marker=dict(color=THEME["accent2"], size=11))], frames=frames)
    fig.update_layout(updatemenus=[play_menu(90)],
                      xaxis=dict(range=[0, 25], title="Month"),
                      yaxis=dict(range=[yv.min() - 12, yv.max() + 12], title="Sales (k units)"))
    plotly_theme(fig, title="Press ▶ — every row becomes one point")
    show_fig(fig)


teach(step=5, total=TOTAL_SCENES, kicker="Visualize", title="From Table to Picture",
      question="How do rows of numbers become something we can see?",
      narration="Numbers in a table hide their story. Watch each row become a point on a chart: the month runs "
                "along the bottom, the sales up the side. Suddenly the pattern is obvious.",
      takeaway="Every row of data becomes one point on the scatter plot — and the picture reveals the trend "
               "instantly.",
      body=scene_body)


In [ ]:
# @title 🎬 Scene 6 · Finding the Relationship    ▶ run me
# ============================================================
#  ONE lesson: month & sales move together — measured by correlation.
# ============================================================
def scene_body():
    strength = abs(CORR) * 100
    show_html('<div class="mlv-wrap"><div class="mlv-card" style="animation-delay:.05s">'
              '<h4>Relationship strength</h4>'
              '<div class="mlv-sub">The points march upward together. We measure that with <b>correlation</b> — '
              'how tightly two variables move as one.</div>'
              f'<div class="mlv-meter"><div style="--w:{strength:.0f}%"></div></div>'
              f'<div class="mlv-sub" style="margin-top:6px">Correlation = '
              f'<b style="color:#FFD166">{CORR:.2f}</b> — a very strong positive relationship.</div></div></div>')
    x, yv = df["Month"].values, df["Sales"].values
    coeffs = np.polyfit(x, yv, 1)
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=x, y=yv, mode="markers", name="Data",
                             marker=dict(color=THEME["accent2"], size=10)))
    fig.add_trace(go.Scatter(x=x, y=np.polyval(coeffs, x), mode="lines", name="Upward trend",
                             line=dict(color=THEME["gold"], width=3, dash="dot")))
    plotly_theme(fig, title=f"As Month ↑, Sales ↑ — correlation ≈ {CORR:.2f}")
    show_fig(fig)


teach(step=6, total=TOTAL_SCENES, kicker="Relationships", title="Finding the Relationship",
      question="Do month and sales actually move together?",
      narration="Look closely: as the months increase, the sales rise with them, almost in lockstep. We measure "
                "that togetherness with correlation. Here it's around zero point nine eight — an exceptionally "
                "strong, positive relationship.",
      takeaway="Month and Sales rise together with a correlation near 0.98 — strong enough that a straight line "
               "will model them well.",
      body=scene_body)


In [ ]:
# @title 🎬 Scene 7 · Choosing the Feature    ▶ run me
# ============================================================
#  ONE lesson: the feature (input) is Month. Everything else fades.
# ============================================================
def scene_body():
    show_html('<div class="mlv-wrap"><div class="mlv-sub">A <b>feature</b> is what we feed the model — the clue '
              'it uses to make a guess. Here, that clue is <b>Month</b>.</div></div>')
    show_html(data_table(df, highlight="Month", fade_others=True))
    show_html(grid(info_card("Feature = Month",
              "Everything else fades away. The model’s only input is the month number. In real projects you "
              "might have many features (price, ads, season…).", 0.1)))


teach(step=7, total=TOTAL_SCENES, kicker="Feature Selection", title="Choosing the Feature",
      question="What will we feed into the model?",
      narration="A feature is the input — the clue the model uses to make its guess. In our problem the feature "
                "is the month number. Everything else fades away so we can focus on it.",
      takeaway="The feature (input) is Month — the single clue the model uses to predict sales.",
      body=scene_body)


In [ ]:
# @title 🎬 Scene 8 · Choosing the Target    ▶ run me
# ============================================================
#  ONE lesson: the target (output) is Sales — what we predict.
# ============================================================
def scene_body():
    show_html('<div class="mlv-wrap"><div class="mlv-sub">The <b>target</b> is the answer we want the model to '
              'produce. Here it is <b>Sales</b>.</div></div>')
    show_html(data_table(df, highlight="Sales", fade_others=True))
    show_html(grid(info_card("Target = Sales",
              "This is the number the business needs predicted. The model learns the relationship "
              "<b>Month → Sales</b> so it can fill in months it has never seen.", 0.1)))


teach(step=8, total=TOTAL_SCENES, kicker="Target Selection", title="Choosing the Target",
      question="What is the model trying to predict?",
      narration="The target is the answer we want back. Here it's the sales figure. The model's whole job is to "
                "learn the link from month to sales, so it can predict months it has never seen.",
      takeaway="The target (output) is Sales — the number the business needs the model to predict.",
      body=scene_body)


In [ ]:
# @title 🎬 Scene 9 · Train / Test Split    ▶ run me
# ============================================================
#  ONE lesson: learn on most of the data, keep some back as an exam.
# ============================================================
def scene_body():
    show_html('<div class="mlv-wrap"><div class="mlv-flow">'
              '<div class="mlv-box">Full dataset<small>24 months</small></div>'
              '<div class="mlv-arrow">➜</div>'
              f'<div class="mlv-box" style="border-color:#00D1B2">Training set<small>{len(y_train)} months · 75%</small></div>'
              f'<div class="mlv-box" style="border-color:#FFD166">Testing set<small>{len(y_test)} months · 25%</small></div>'
              '</div></div>')
    show_html(grid(
        info_card("Why split?", "The model <b>learns</b> from the training set…", 0.1),
        info_card("The exam", "…then we <b>test</b> it on months it never saw, to check it truly learned the "
                  "pattern instead of memorising.", 0.2),
    ))
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=X_train[:, 0], y=y_train, mode="markers", name="Training (75%)",
                             marker=dict(color=THEME["accent2"], size=11, symbol="circle")))
    fig.add_trace(go.Scatter(x=X_test[:, 0], y=y_test, mode="markers", name="Testing (25%)",
                             marker=dict(color=THEME["gold"], size=13, symbol="diamond")))
    fig.update_xaxes(title="Month"); fig.update_yaxes(title="Sales (k units)")
    plotly_theme(fig, title="The data, split into learning vs. exam sets")
    show_fig(fig)


teach(step=9, total=TOTAL_SCENES, kicker="Train / Test Split", title="Splitting the Data",
      question="How do we know the model actually learned?",
      narration="We hold some data back. The model learns from the training set — the larger share — and is "
                "then tested on months it never saw. That test is how we catch a model that only memorised "
                "instead of truly learning.",
      takeaway="We train on most of the data and keep a testing set aside to fairly check the model on unseen "
               "months.",
      body=scene_body)


In [ ]:
# @title 🔬 Scene 10 · Inside the Model — watch it learn, step by step    ▶ run me
# ==================================================================
#  A purely VISUAL, animated walk-through — no walls of text. Two
#  animations you press ▶ on and simply watch:
#
#   ① Forward pass as a NETWORK DIAGRAM (left → right):
#        INPUT box  --→  [ weight × ]  --→  [ bias + ]  --→  PREDICTION ŷ
#        then ŷ is compared to the ACTUAL value, a red ERROR bar opens,
#        and a feedback arrow says "update weight & bias".
#        Captions explain WHAT the weight/bias are and WHY they exist.
#
#   ② Gradient descent — the line starts flat & wrong, and step by
#        step tilts into the best fit while every red error bar
#        shrinks and a live loss curve slides downhill.
# ==================================================================
def scene_body():
    C = THEME
    x_arr = df["Month"].values.astype(float)
    y_arr = df["Sales"].values.astype(float)

    # =============================================================
    #  ANIMATION ①  —  one input's journey through the model
    #                  (input → × weight → + bias → prediction → error)
    # =============================================================
    m      = 12
    v_in   = float(m)
    v_w    = SLOPE * m                 # value after × weight
    v_pred = INTERCEPT + SLOPE * m     # value after + bias  (prediction)
    actual = float(df.loc[df["Month"] == m, "Sales"].iloc[0])
    err    = actual - v_pred

    def lerp(a, b, t):
        return a + (b - a) * t

    # ---- fixed geometry of the diagram (data coordinates) --------
    IN_C, W_C, B_C, P_C = (1.3, 5.0), (4.9, 5.0), (7.1, 5.0), (9.7, 5.0)
    A_C  = (11.6, 7.2)                 # actual node
    ERRX = 11.6                        # error bar x
    EB0, EB1 = 2.2, 5.8                # error bar bottom / max top

    # ---- tiny trace factories -----------------------------------
    def empty(mode="lines"):
        return go.Scatter(x=[], y=[], mode=mode, showlegend=False, hoverinfo="skip")

    def box(cx, cy, w, h, color, width=2.5):
        x0, x1, y0, y1 = cx - w / 2, cx + w / 2, cy - h / 2, cy + h / 2
        return go.Scatter(x=[x0, x1, x1, x0, x0], y=[y0, y0, y1, y1, y0], mode="lines",
                          line=dict(color=color, width=width), showlegend=False, hoverinfo="skip")

    def bar(cx, y_top, color):
        x0, x1 = cx - 0.28, cx + 0.28
        return go.Scatter(x=[x0, x1, x1, x0, x0], y=[EB0, EB0, y_top, y_top, EB0], mode="lines",
                          fill="toself", fillcolor="rgba(255,107,107,.35)",
                          line=dict(color=color, width=2), showlegend=False, hoverinfo="skip")

    def node(cx, cy, val, active=False, border=None, fill=None, txtcol=None, size=58):
        border = border or (C["gold"] if active else C["accent2"])
        return go.Scatter(x=[cx], y=[cy], mode="markers+text",
                          marker=dict(size=size, color=fill or C["panel2"],
                                      line=dict(color=border, width=3)),
                          text=[f"{val:.0f}" if val is not None else ""],
                          textposition="middle center",
                          textfont=dict(color=txtcol or C["gold"], size=17, family="Inter"),
                          showlegend=False, hoverinfo="skip")

    def arrow(x0, y0, x1, y1, color, frac, dash=None, width=4):
        if not frac or frac <= 0:
            return empty("lines+markers")
        xt, yt = x0 + (x1 - x0) * frac, y0 + (y1 - y0) * frac
        return go.Scatter(x=[x0, xt], y=[y0, yt], mode="lines+markers",
                          line=dict(color=color, width=width, dash=dash),
                          marker=dict(symbol="arrow", size=15, color=color,
                                      angleref="previous", opacity=[0, 1]),
                          showlegend=False, hoverinfo="skip")

    def static_labels():
        lx = [IN_C[0], 6.0,   W_C[0],       B_C[0],     P_C[0],       A_C[0],   ERRX]
        ly = [6.55,    7.95,  3.7,          3.7,        3.7,          8.25,     1.5]
        lt = ["INPUT", "MODEL", "weight (w)", "bias (b)", "PREDICTION", "ACTUAL", "ERROR"]
        lc = [C["accent"], C["gold"], C["muted"], C["muted"], C["gold"], C["accent2"], C["bad"]]
        return go.Scatter(x=lx, y=ly, mode="text", text=lt,
                          textfont=dict(color=lc, size=13), showlegend=False, hoverinfo="skip")

    # ---- the 17 trace slots for one frame -----------------------
    def render(st):
        # 11 · error bar height
        etop = EB0 + (EB1 - EB0) * st["err_frac"]
        # 12 · error labels
        elx, ely, elt, els = [], [], [], []
        if st["err_show"]:
            elx += [ERRX]; ely += [(EB0 + etop) / 2]; elt += [f"{err:+.0f}"]; els += [16]
        if st["fb"] > 0:
            elx += [9.6]; ely += [3.0]; elt += ["update weight & bias"]; els += [11]
        # 14 · operation labels above active gates
        ox, oy, ot = [], [], []
        if st["w_show"]:
            ox += [W_C[0]]; oy += [6.15]; ot += [f"× {SLOPE:.2f}"]
        if st["b_show"]:
            ox += [B_C[0]]; oy += [6.15]; ot += [f"+ {INTERCEPT:.1f}"]

        return [
            box(6.0, 5.0, 4.8, 5.2, C["gold"]),                                             # 0 model box
            box(IN_C[0], IN_C[1], 1.9, 2.5, C["accent"]) if st["in_show"] else empty(),     # 1 input box
            go.Scatter(x=[IN_C[0]], y=[IN_C[1]], mode="text", text=[f"{st['in_val']:.0f}"],
                       textfont=dict(color=C["text"], size=26), showlegend=False,
                       hoverinfo="skip") if st["in_show"] else empty("text"),               # 2 input value
            arrow(2.35, 5.0, 4.45, 5.0, C["accent"], st["aIn"]),                            # 3 violet arrow
            node(*W_C, st["w_val"], active=st["w_active"]) if st["w_show"] else empty("markers+text"),  # 4 weight node
            arrow(5.35, 5.0, 6.55, 5.0, C["accent2"], st["aW1"]),                           # 5 teal arrow w→b
            node(*B_C, st["b_val"], active=st["b_active"]) if st["b_show"] else empty("markers+text"),  # 6 bias node
            arrow(7.55, 5.0, 9.15, 5.0, C["accent2"], st["aW2"]),                           # 7 teal arrow b→pred
            node(*P_C, st["p_val"], border=C["gold"], fill=C["bad"], txtcol="#fff")
                if st["p_show"] else empty("markers+text"),                                 # 8 prediction node
            arrow(10.05, 5.3, 11.2, 6.9, C["gold"], st["aGold"]),                           # 9 gold arrow pred→actual
            node(*A_C, actual, border=C["accent2"], fill=C["accent2"], txtcol="#0F1226", size=52)
                if st["act_show"] else empty("markers+text"),                               # 10 actual node
            bar(ERRX, etop, C["bad"]) if st["err_show"] else empty(),                       # 11 error bar
            go.Scatter(x=elx, y=ely, mode="text", text=elt,
                       textfont=dict(color=C["bad"], size=els), showlegend=False,
                       hoverinfo="skip") if elx else empty("text"),                         # 12 error labels
            arrow(11.4, 2.6, 8.5, 6.6, C["bad"], st["fb"], dash="dot", width=3),            # 13 red feedback arrow
            go.Scatter(x=ox, y=oy, mode="text", text=ot,
                       textfont=dict(color=C["gold"], size=17), showlegend=False,
                       hoverinfo="skip") if ox else empty("text"),                          # 14 op labels
            static_labels(),                                                                # 15 static labels
            go.Scatter(x=[0.3, 0.3], y=[9.15, 8.55], mode="text",
                       text=[st["cap"], st["sub"]], textposition="middle right",
                       textfont=dict(color=[C["text"], C["muted"]], size=[17, 13.5]),
                       showlegend=False, hoverinfo="skip"),                                 # 16 caption + sub
        ]

    # ---- build the storyboard (list of states) ------------------
    states = []
    S = dict(in_show=False, in_val=0.0, aIn=0.0,
             w_show=False, w_active=False, w_val=None, aW1=0.0,
             b_show=False, b_active=False, b_val=None, aW2=0.0,
             p_show=False, p_val=None, aGold=0.0, act_show=False,
             err_show=False, err_frac=0.0, fb=0.0, cap="", sub="")

    def snap():
        states.append(dict(S))

    def hold(n=1):
        for _ in range(n):
            snap()

    S["cap"] = "A single point's journey through the model"
    S["sub"] = "press ▶ and watch — no equations"; hold(2)

    # ① input enters
    S["in_show"] = True; S["cap"] = f"①  The input enters  —  Month {m:.0f}"
    S["sub"] = "this is the number we feed the model"
    for s in range(7):
        S["in_val"] = lerp(0, v_in, s / 6); snap()
    S["cap"] = "…  it flows into the model"; S["sub"] = ""
    for s in range(5):
        S["aIn"] = (s + 1) / 5; snap()

    # ② weight gate
    S["w_show"] = True; S["w_active"] = True
    S["cap"] = "②  First gate — the WEIGHT (w)"
    S["sub"] = f"the weight MULTIPLIES the input. It sets the slope: how fast sales rise per month.   × {SLOPE:.2f}"
    for s in range(9):
        S["w_val"] = lerp(v_in, v_w, s / 8); snap()
    hold(1); S["sub"] = ""
    for s in range(5):
        S["aW1"] = (s + 1) / 5; snap()

    # ③ bias gate
    S["b_show"] = True; S["b_active"] = True
    S["cap"] = "③  Second gate — the BIAS (b)"
    S["sub"] = f"the bias is ADDED. It shifts the line up/down: the base level when month = 0.   + {INTERCEPT:.1f}"
    for s in range(9):
        S["b_val"] = lerp(v_w, v_pred, s / 8); snap()
    hold(1); S["sub"] = ""
    for s in range(5):
        S["aW2"] = (s + 1) / 5; snap()

    # ④ prediction
    S["p_show"] = True; S["p_val"] = v_pred
    S["cap"] = f"④  Out comes the PREDICTION   ŷ = {v_pred:.0f}"
    S["sub"] = "input, stretched by the weight and lifted by the bias"; hold(4)

    # ⑤ compare to actual
    S["act_show"] = True
    S["cap"] = f"⑤  Compare with the real ACTUAL value = {actual:.0f}"; S["sub"] = ""
    for s in range(6):
        S["aGold"] = (s + 1) / 6; snap()

    # ⑥ error gap
    S["err_show"] = True
    S["cap"] = f"⑥  The gap is the ERROR = actual {actual:.0f} − predicted {v_pred:.0f} = {err:+.0f}"
    for s in range(7):
        S["err_frac"] = (s + 1) / 7; snap()
    hold(2)

    # ⑦ feedback → gradient descent
    S["cap"] = "⑦  Shrink that error — nudge the WEIGHT & BIAS"
    S["sub"] = "that repeated nudge is GRADIENT DESCENT  →  watch it below 👇"
    for s in range(6):
        S["fb"] = (s + 1) / 6; snap()
    hold(4)

    frames1 = [go.Frame(data=render(st), name=str(i)) for i, st in enumerate(states)]
    fig1 = go.Figure(data=render(states[0]), frames=frames1)
    fig1.update_layout(
        updatemenus=[play_menu(70)],
        sliders=[dict(active=0, x=0.05, y=-0.02, len=0.9, pad=dict(t=26),
                      currentvalue=dict(visible=False),
                      steps=[dict(method="animate", label="",
                                  args=[[str(i)], dict(mode="immediate",
                                        frame=dict(duration=0, redraw=True),
                                        transition=dict(duration=0))]) for i in range(len(frames1))])],
    )
    plotly_theme(fig1, height=560,
                 title="① One point's journey — input → × weight → + bias → prediction → error")
    fig1.update_xaxes(visible=False, range=[-0.2, 13.0])
    fig1.update_yaxes(visible=False, range=[0.8, 9.7])
    show_fig(fig1)

    show_html('<div class="mlv-wrap"><div class="mlv-sub" style="margin:14px 0 2px;font-size:15px">'
              'That was <b>one</b> point. Now the model does it for <b>every</b> point at once and, '
              'step by step, nudges the weight &amp; bias to make all those red error bars as small as '
              'possible — that search is <b>gradient descent</b>. Press ▶ below. 👇</div></div>')

    # =============================================================
    #  ANIMATION ②  —  gradient descent shrinks the error, step by step
    # =============================================================
    xm, xs = x_arr.mean(), x_arr.std()
    xz = (x_arr - xm) / xs                       # normalise for a smooth, stable descent
    a, c, lr = 0.0, 0.0, 0.15                    # start from a deliberately terrible flat line (y = 0)
    traj = []
    for _ in range(42):
        pr = a * xz + c
        e = pr - y_arr
        traj.append(dict(so=a / xs, io=c - a * xm / xs, loss=float(np.mean(e ** 2))))
        a -= lr * 2 * np.mean(e * xz)
        c -= lr * 2 * np.mean(e)
    max_loss = traj[0]["loss"]
    y_top = float(y_arr.max()) + 18

    def gd_frame(k):
        st = traj[k]
        so, io = st["so"], st["io"]
        xln = np.array([0.0, 25.0])
        yln = io + so * xln
        pred_pts = io + so * x_arr
        rx, ry = [], []
        for xi, yi, pi in zip(x_arr, y_arr, pred_pts):
            rx += [xi, xi, None]
            ry += [yi, pi, None]
        lk = [t["loss"] for t in traj[:k + 1]]
        ik = list(range(1, k + 2))
        return [
            # 0 · red error bars (residuals) from each point to the line
            go.Scatter(x=rx, y=ry, mode="lines", line=dict(color="rgba(255,107,107,.75)", width=1.6),
                       showlegend=False),
            # 1 · the data points
            go.Scatter(x=x_arr, y=y_arr, mode="markers",
                       marker=dict(color="rgba(0,209,178,.9)", size=7), showlegend=False),
            # 2 · the current line
            go.Scatter(x=xln, y=yln, mode="lines", line=dict(color=C["gold"], width=4), showlegend=False),
            # 3 · live readout (left top)
            go.Scatter(x=[0.6], y=[y_top - 3], mode="text",
                       text=[f"step {k + 1}   ·   slope {so:.2f}   ·   bias {io:.1f}"
                             f"   ·   total error {st['loss']:,.0f}"],
                       textposition="middle right", textfont=dict(color=C["text"], size=14),
                       showlegend=False),
            # 4 · loss curve so far (right panel)
            go.Scatter(x=ik, y=lk, mode="lines", line=dict(color=C["accent2"], width=3),
                       fill="tozeroy", fillcolor="rgba(0,209,178,.15)",
                       xaxis="x2", yaxis="y2", showlegend=False),
            # 5 · moving marker at the current loss (right panel)
            go.Scatter(x=[ik[-1]], y=[lk[-1]], mode="markers",
                       marker=dict(color=C["bad"], size=11, line=dict(color="white", width=1)),
                       xaxis="x2", yaxis="y2", showlegend=False),
        ]

    g_frames = [go.Frame(data=gd_frame(k), name=str(k)) for k in range(len(traj))]
    fig2 = go.Figure(data=gd_frame(0), frames=g_frames)
    fig2.update_layout(
        updatemenus=[play_menu(120)],
        sliders=[dict(active=0, x=0.05, y=-0.06, len=0.9, pad=dict(t=28),
                      currentvalue=dict(visible=False),
                      steps=[dict(method="animate", label="",
                                  args=[[str(k)], dict(mode="immediate",
                                        frame=dict(duration=0, redraw=True),
                                        transition=dict(duration=0))]) for k in range(len(g_frames))])],
        annotations=[
            dict(x=0.30, y=1.06, xref="paper", yref="paper", showarrow=False,
                 text="the line tilts to fit — red gaps = error", font=dict(color=C["muted"], size=12)),
            dict(x=0.88, y=1.06, xref="paper", yref="paper", showarrow=False,
                 text="total error sliding downhill ↓", font=dict(color=C["muted"], size=12)),
        ],
        xaxis=dict(domain=[0.0, 0.60], range=[0, 25.5], title="Month",
                   gridcolor=C["grid"], zeroline=False),
        yaxis=dict(domain=[0.0, 1.0], range=[0, y_top], title="Sales (k units)",
                   gridcolor=C["grid"], zeroline=False),
        xaxis2=dict(domain=[0.70, 1.0], range=[0.5, len(traj) + 0.5], title="learning step",
                    anchor="y2", gridcolor=C["grid"], zeroline=False),
        yaxis2=dict(domain=[0.0, 1.0], range=[0, max_loss * 1.05], title="total error (loss)",
                    anchor="x2", gridcolor=C["grid"], zeroline=False),
    )
    plotly_theme(fig2, height=540,
                 title="② Gradient descent — watch every red error bar shrink, step by step")
    show_fig(fig2)

    # one short, honest footnote (not a written lesson — just a caveat)
    show_html('<div class="mlv-wrap"><div class="mlv-takeaway" style="border-color:#00D1B2;'
              'border-left-color:#00D1B2;background:rgba(0,209,178,.1)">'
              'ℹ️ <b>Note:</b> the descent above is an <i>educational animation</i> of the idea — the intuition '
              'is exactly right, but scikit-learn actually solves for this best line directly with linear '
              'algebra instead of stepping downhill.</div></div>')


teach(step=10, total=TOTAL_SCENES, kicker="Inside the Model",
      title="Inside the Model — watch it learn, step by step",
      question="What actually happens to data inside the model, and how does it learn to be less wrong?",
      narration="Let's watch it happen, no equations. One input, the number twelve, enters the model. At the first "
                "gate it meets the weight and gets multiplied — the weight sets the slope, how fast sales rise each "
                "month — so twelve becomes fifty-one. At the second gate the bias is added — the bias sets the base "
                "level, the starting point when the month is zero — lifting it up to the prediction. Beside it the "
                "real value rises, and the red bar between them is the error. To shrink that error the model nudges "
                "the weight and the bias, again and again. That repeated nudge is gradient descent: in the second "
                "animation the line starts flat and wrong, then tilts until every red error bar is as short as it "
                "can be, and the total error slides downhill until it flattens.",
      takeaway="A prediction is just the input multiplied by the weight (the slope) and lifted by the bias (the "
               "base level). The gap to reality is the error — and gradient descent keeps nudging the weight and "
               "bias, step by step, until that error is as small as it can get. That downhill search is what "
               "“learning” looks like.",
      body=scene_body)


In [ ]:
# @title 🎯 Scene 11 · Finding the Best-Fit Line — YOU try it!    ▶ run me
# ==================================================================
#  ONE lesson: the "best" line is the one with the smallest total error.
#  Drag the sliders, watch the red error bars and the Sum of Squared
#  Errors (SSE) change live, then reveal the true best fit.
# ==================================================================
def scene_body():
    x = df["Month"].values.astype(float)
    yv = df["Sales"].values.astype(float)
    best_sse = float(np.sum((yv - (INTERCEPT + SLOPE * x)) ** 2))   # the target to beat

    show_html('<div class="mlv-wrap"><div class="mlv-sub">Every red bar is one point\'s <b>error</b> — the gap '
              'to your line. Add up all those gaps <i>squared</i> and you get the <b>SSE</b>. Your mission: '
              'move the sliders to make SSE as <b>small as possible</b>.</div></div>')

    slope_sl = widgets.FloatSlider(value=2.0, min=0.0, max=8.0, step=0.1, description="Slope",
                                   continuous_update=False, readout_format=".1f",
                                   style={"description_width": "70px"},
                                   layout=widgets.Layout(width="46%"))
    inter_sl = widgets.FloatSlider(value=90.0, min=0.0, max=140.0, step=1.0, description="Intercept",
                                   continuous_update=False, readout_format=".0f",
                                   style={"description_width": "70px"},
                                   layout=widgets.Layout(width="46%"))
    reveal_btn = widgets.Button(description="✨ Reveal the true best fit",
                                layout=widgets.Layout(width="230px"))
    reveal_btn.style.button_color = THEME["gold"]
    status = widgets.Output()
    plot = widgets.Output()

    def draw(*_):
        s, b = slope_sl.value, inter_sl.value
        yhat = b + s * x
        sse = float(np.sum((yv - yhat) ** 2))

        # Residual segments as a single trace (None separates each little bar).
        rx, ry = [], []
        for xi, yi, yh in zip(x, yv, yhat):
            rx += [xi, xi, None]
            ry += [yi, yh, None]

        fig = go.Figure()
        fig.add_trace(go.Scatter(x=rx, y=ry, mode="lines", name="Errors",
                                 line=dict(color=THEME["bad"], width=1.6)))
        fig.add_trace(go.Scatter(x=x, y=yv, mode="markers", name="Data",
                                 marker=dict(color=THEME["accent2"], size=9)))
        ends = np.array([x.min(), x.max()])
        fig.add_trace(go.Scatter(x=ends, y=b + s * ends, mode="lines", name="Your line",
                                 line=dict(color=THEME["gold"], width=3)))
        plotly_theme(fig, title="Drag the sliders — shrink the red bars")
        plot.clear_output(wait=True)
        with plot:
            show_fig(fig)

        # Live score + how close to optimal.
        ratio = best_sse / sse if sse > 0 else 1.0          # 1.0 = perfect
        pct = max(0.0, min(100.0, ratio * 100))
        verdict = ("🏆 That's the best fit!" if sse <= best_sse * 1.02
                   else "🔥 Very close!" if pct > 85
                   else "👍 Getting warmer…" if pct > 55
                   else "❄️ Keep going — big errors remain")
        status.clear_output(wait=True)
        with status:
            show_html(
                '<div class="mlv-wrap"><div class="mlv-card" style="animation:none">'
                f'<h4>Your score</h4>'
                f'<div class="mlv-stat" style="color:#FF6B6B">SSE = {sse:,.0f}</div>'
                f'<div class="mlv-sub">Best possible ≈ <b style="color:#FFD166">{best_sse:,.0f}</b> &nbsp;·&nbsp; {verdict}</div>'
                f'<div class="mlv-meter"><div style="--w:{pct:.0f}%;animation:none;width:{pct:.0f}%"></div></div>'
                '</div></div>')

    def reveal(_):
        # Snap the sliders to the model's learned parameters.
        slope_sl.value = round(SLOPE, 1)
        inter_sl.value = round(INTERCEPT, 0)

    slope_sl.observe(draw, names="value")
    inter_sl.observe(draw, names="value")
    reveal_btn.on_click(reveal)

    display(widgets.HBox([slope_sl, inter_sl], layout=widgets.Layout(flex_flow="row wrap")))
    display(reveal_btn)
    display(status, plot)
    draw()


teach(step=11, total=TOTAL_SCENES, kicker="Best Fit", title="Finding the Best-Fit Line",
      question="Out of every possible line, which one is 'best' — and how do we score it?",
      narration="Now it's your turn. Each red bar is the error for one point — how far the real value sits from "
                "your line. Square every error, add them up, and you get the Sum of Squared Errors. The best line "
                "is simply the one that makes this number as small as possible. Try the sliders, then reveal the "
                "line the model found.",
      takeaway="The best-fit line is the one with the smallest Sum of Squared Errors. Linear Regression finds it "
               "automatically — you just felt how hard that search is by hand.",
      body=scene_body)


In [ ]:
# @title 🎬 Scene 12 · Watch the Model Learn    ▶ run me
# ============================================================
#  ONE lesson: the line improves automatically as its loss falls.
#  Press ▶ — slope, intercept and loss (MSE) update together.
# ============================================================
def scene_body():
    show_html('<div class="mlv-wrap"><div class="mlv-sub">You minimised the error by hand. Now watch the model '
              'do it automatically: the line starts poorly, then the equation sharpens and the <b>loss</b> '
              '(average error) drops to its smallest value. Press ▶.</div></div>')

    x = df["Month"].values.astype(float)
    yv = df["Sales"].values.astype(float)
    s0, b0 = 0.6, float(yv.min())            # a deliberately poor starting line

    frames = []
    for j, t in enumerate(np.linspace(0, 1, 28)):
        s = s0 + (SLOPE - s0) * t            # interpolate toward the learned params
        b = b0 + (INTERCEPT - b0) * t
        yl = b + s * x
        loss = float(np.mean((yv - yl) ** 2))
        ann = [dict(x=0.98, y=0.05, xref="paper", yref="paper", align="right", showarrow=False,
                    bgcolor="rgba(15,18,38,.75)", bordercolor=THEME["gold"], borderpad=8,
                    font=dict(size=15, color=THEME["gold"]),
                    text=f"ŷ = {b:.1f} + {s:.2f}·x<br>Loss (MSE) = {loss:,.0f}")]
        frames.append(go.Frame(
            data=[go.Scatter(x=x, y=yv, mode="markers", marker=dict(color=THEME["accent2"], size=9), name="Actual"),
                  go.Scatter(x=x, y=yl, mode="lines", line=dict(color=THEME["gold"], width=4), name="Model line")],
            layout=dict(annotations=ann), name=str(j)))

    fig = go.Figure(data=list(frames[0].data), frames=frames, layout=frames[0].layout)
    fig.update_layout(updatemenus=[play_menu(150)],
                      xaxis=dict(range=[0, 25], title="Month"),
                      yaxis=dict(range=[yv.min() - 15, yv.max() + 15], title="Sales (k units)"))
    plotly_theme(fig, title="Watch the line learn — slope, intercept & loss move together")
    show_fig(fig)
    show_html(grid(info_card("Final learned equation",
              f'Sales ≈ <b style="color:#FFD166">{INTERCEPT:.1f}</b> + '
              f'<b style="color:#FFD166">{SLOPE:.2f}</b> × Month', 0.1)))


teach(step=12, total=TOTAL_SCENES, kicker="Watch it Learn", title="Watch the Model Learn",
      question="Can we see the line improve automatically in real time?",
      narration="Watch it on our real data. The line starts poorly, but with every step the slope and intercept "
                "adjust, the equation updates, and the loss — the total error — drops toward its smallest "
                "possible value.",
      takeaway=f"As the line learns, its equation sharpens and the loss falls — settling on "
               f"Sales ≈ {INTERCEPT:.0f} + {SLOPE:.1f} × Month.",
      body=scene_body)


In [ ]:
# @title 🎬 Scene 13 · Testing on Unseen Data    ▶ run me
# ============================================================
#  ONE lesson: prove the model learned by scoring it on the exam set.
# ============================================================
def scene_body():
    show_html(grid(
        stat_card("R² score", f"{R2:.2f}", "1.0 = perfect fit", 0.05, THEME["good"]),
        stat_card("MAE", f"{MAE:.1f}k", "avg error per month", 0.15, THEME["gold"]),
        stat_card("RMSE", f"{RMSE:.1f}k", "typical error size", 0.25, THEME["accent2"]),
    ))
    show_html('<div class="mlv-wrap"><div class="mlv-sub">The model meets months it has never seen. Its '
              'predictions land close to reality — proof it learned the <b>pattern</b>, not the noise.</div></div>')
    order = np.argsort(X_test[:, 0])
    months = X_test[order, 0]
    actual = y_test[order]
    pred = _y_pred_test[order]
    fig = go.Figure()
    fig.add_trace(go.Bar(x=[f"M{int(m)}" for m in months], y=actual, name="Actual",
                         marker_color=THEME["accent2"]))
    fig.add_trace(go.Bar(x=[f"M{int(m)}" for m in months], y=pred, name="Predicted",
                         marker_color=THEME["gold"]))
    fig.update_layout(barmode="group")
    fig.update_yaxes(title="Sales (k units)")
    plotly_theme(fig, title="On unseen months: predicted vs. actual")
    show_fig(fig)


teach(step=13, total=TOTAL_SCENES, kicker="Testing", title="Testing the Model",
      question="How good is the model on data it never saw?",
      narration="Time for the exam. We show the model the months we held back and compare its predictions to "
                "reality. The bars line up closely, and the scores confirm it: the model generalised well.",
      takeaway="On unseen months the predictions land close to reality — a high R² and low error mean the model "
               "truly learned the pattern.",
      body=scene_body)


In [ ]:
# @title 🎬 Scene 14 · Forecast the Future (interactive)    ▶ run me
# ============================================================
#  ONE lesson: feed any future month, get an instant prediction.
# ============================================================
def scene_body():
    show_html('<div class="mlv-wrap"><div class="mlv-sub">This is what the business wanted. Drag the slider to '
              'any future month and the prediction updates <b>instantly</b>.</div></div>')
    slider = widgets.IntSlider(value=25, min=25, max=48, step=1, description="Future month:",
                               continuous_update=False, style={"description_width": "initial"},
                               layout=widgets.Layout(width="70%"))
    out = widgets.Output()

    def update(*_):
        month = slider.value
        pred = predict_sales(month)
        xs = np.arange(1, month + 1)
        fig = go.Figure()
        fig.add_trace(go.Scatter(x=df["Month"], y=df["Sales"], mode="markers", name="History",
                                 marker=dict(color=THEME["accent2"], size=9)))
        fig.add_trace(go.Scatter(x=xs, y=INTERCEPT + SLOPE * xs, mode="lines", name="Trend line",
                                 line=dict(color=THEME["gold"], width=3)))
        fig.add_trace(go.Scatter(x=[month], y=[pred], mode="markers+text", name="Forecast",
                                 marker=dict(color=THEME["bad"], size=18, symbol="star"),
                                 text=[f"  {pred:.0f}k"], textposition="middle right",
                                 textfont=dict(color=THEME["gold"], size=16)))
        fig.update_xaxes(title="Month"); fig.update_yaxes(title="Sales (k units)")
        plotly_theme(fig, title=f"Month {month} → predicted {pred:.1f}k units")
        out.clear_output(wait=True)
        with out:
            show_fig(fig)

    slider.observe(update, names="value")
    display(slider, out)
    update()


teach(step=14, total=TOTAL_SCENES, kicker="Forecast", title="Forecast the Future",
      question="What will sales be in a month we've never seen?",
      narration="This is the payoff the business asked for. Move the slider to any future month and the model "
                "instantly predicts the sales, placing a star exactly where the trend line points.",
      takeaway="Feed the model any future month and it returns an instant sales prediction — the business's "
               "original question, answered.",
      body=scene_body)


In [ ]:
# @title 🎬 Scene 15 · Business Recommendation Engine    ▶ run me
# ============================================================
#  ONE lesson: a prediction is only useful if it drives a decision.
# ============================================================
def scene_body():
    nxt = 25
    pred = predict_sales(nxt)
    buffer = pred * 1.12
    show_html('<div class="mlv-wrap"><div class="mlv-flow">'
              f'<div class="mlv-box model">Forecast<small>{pred:.0f}k units (M{nxt})</small></div>'
              '<div class="mlv-arrow">➜</div>'
              '<div class="mlv-box">Decisions<small>auto-generated</small></div></div></div>')
    show_html(grid(
        info_card("📦 Inventory",
                  f"Stock ~<b>{buffer:.0f}k units</b> (forecast + 12% safety buffer) to meet demand without "
                  "over-ordering.", 0.05),
        info_card("📣 Marketing",
                  "Demand is rising on its own — keep spend <b>steady</b> and protect margins; push harder only "
                  "if you want to beat the trend.", 0.15),
        info_card("💰 Revenue outlook",
                  f"With ~{SLOPE:.1f}k units added every month, expect <b>continued growth</b> into next quarter.",
                  0.25),
        info_card("⚠️ Business risk",
                  "<b>Low–moderate.</b> The trend is strong and steady, but a straight line assumes it "
                  "<i>keeps</i> going — watch for market saturation.", 0.35),
    ))


teach(step=15, total=TOTAL_SCENES, kicker="Recommendations", title="Business Recommendation Engine",
      question="What should ABC Electronics actually DO with this prediction?",
      narration="A prediction is only useful if it drives a decision. From the forecast we generate concrete "
                "advice: how much inventory to stock, how to steer marketing, the revenue outlook, and the "
                "business risk.",
      takeaway="A good model doesn't stop at a number — it feeds inventory, marketing, revenue and risk "
               "decisions.",
      body=scene_body)


In [ ]:
# @title 🎬 Scene 16 · Explain Why    ▶ run me
# ============================================================
#  ONE lesson: because the model is a line, every prediction is explainable.
# ============================================================
def scene_body():
    nxt = 25
    pred = predict_sales(nxt)
    btn = widgets.Button(description="🔍  Explain Why",
                         layout=widgets.Layout(width="200px", height="42px"))
    btn.style.button_color = THEME["accent"]
    out = widgets.Output()

    explanation = (
        '<div class="mlv-wrap"><div class="mlv-grid">'
        '<div class="mlv-card" style="animation-delay:.05s"><h4>1 · The trend</h4><div class="mlv-sub">'
        f'The dots climbed steadily (correlation ≈ {CORR:.2f}), so a straight line is a sensible model.</div></div>'
        '<div class="mlv-card" style="animation-delay:.15s"><h4>2 · The equation</h4><div class="mlv-sub">'
        f'Sales ≈ <b>{INTERCEPT:.1f}</b> + <b>{SLOPE:.2f}</b> × Month. The intercept is the starting level; the '
        f'slope says sales grow ~{SLOPE:.1f}k every month.</div></div>'
        '<div class="mlv-card" style="animation-delay:.25s"><h4>3 · The prediction</h4><div class="mlv-sub">'
        f'For month {nxt}: {INTERCEPT:.1f} + {SLOPE:.2f}×{nxt} ≈ <b>{pred:.0f}k units</b>.</div></div>'
        '<div class="mlv-card" style="animation-delay:.35s"><h4>4 · The recommendation</h4><div class="mlv-sub">'
        f'Rising demand → stock ahead of the {pred:.0f}k forecast and plan for growth.</div></div>'
        '</div></div>')

    def on_click(_):
        out.clear_output(wait=True)
        with out:
            show_html(explanation)

    btn.on_click(on_click)
    show_html('<div class="mlv-wrap"><div class="mlv-sub">Press the button for a plain-English explanation of '
              'the whole model — trend, equation, prediction and advice.</div></div>')
    display(btn, out)
    on_click(None)   # reveal immediately so the scene is never empty


teach(step=16, total=TOTAL_SCENES, kicker="Explain Why", title="Explain Why",
      question="Can we explain the whole model in plain English?",
      narration="Press Explain Why and the app tells the whole story simply: the trend it found, the equation it "
                "learned, the prediction it makes, and the business recommendation that follows. "
                "Interpretability is a superpower of linear regression.",
      takeaway="Because the model is just a line, every prediction can be explained in one clear sentence — "
               "trend, equation, prediction, action.",
      body=scene_body)


In [ ]:
# @title 🎬 Scene 17 · Summary    ▶ run me
# ============================================================
#  ONE lesson: the big ideas to carry away.
# ============================================================
def scene_body():
    show_html(grid(
        info_card("✅ Advantages",
                  "Simple, fast, fully <b>interpretable</b> — you can read the equation and explain every "
                  "prediction to a manager.", 0.05),
        info_card("⚠️ Limitations",
                  "Assumes a <b>straight-line</b> relationship. Real sales can curve, plateau or spike with "
                  "seasons — then you need richer models.", 0.15),
    ))
    show_html(grid(
        info_card("🏢 Business uses",
                  "Sales & demand forecasting, pricing, budgeting, risk scoring — anywhere one number drives "
                  "another.", 0.25),
        info_card("🎓 Key learnings",
                  "Data → picture → relationship → best-fit line → predict → decide. That loop is the heart of "
                  "supervised machine learning.", 0.35),
    ))
    # Replay the core "learning" visual as a closing recap.
    x = df["Month"].values.astype(float)
    xs = np.arange(1, 25)
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=df["Month"], y=df["Sales"], mode="markers", name="Actual",
                             marker=dict(color=THEME["accent2"], size=9)))
    fig.add_trace(go.Scatter(x=xs, y=INTERCEPT + SLOPE * xs, mode="lines", name="Best-fit line",
                             line=dict(color=THEME["gold"], width=4)))
    plotly_theme(fig, title=f"Everything in one line:  Sales ≈ {INTERCEPT:.1f} + {SLOPE:.2f} × Month")
    show_fig(fig)


teach(step=17, total=TOTAL_SCENES, kicker="Summary", title="What You've Learned",
      question="What are the big ideas to carry away?",
      narration="You've travelled from raw data to a working forecast. You've seen the advantages of linear "
                "regression, its limitations, where businesses use it, and the core loop of supervised learning. "
                "That understanding is yours to keep.",
      takeaway="Data → picture → relationship → best-fit line → prediction → decision. Master that loop and you "
               "understand the foundation of machine learning.",
      body=scene_body)
